# 03_v2 — Build Pages for Generation

Takes the candidate URL pool and:
1. Fetches HTML (100 concurrent, cached to disk)
2. Quality filters (text length, parked page detection, English heuristic)
3. Samples per type to hit distribution targets
4. Takes Playwright screenshots (4 parallel browsers)
5. Saves `pages_for_generation.jsonl`

Fully resumable — cached HTML and screenshots persist across runs.

**Input:** `data/raw/wdc_candidate_urls.jsonl`  
**Output:** `data/processed/pages_for_generation.jsonl`

In [ ]:
!python3 -u ../scripts/build_pages.py

In [ ]:
# Summary
import json
from pathlib import Path
from collections import Counter
from urllib.parse import urlparse

out_path = Path('../data/processed/pages_for_generation.jsonl')
pages = [json.loads(l) for l in out_path.read_text().strip().split('\n')]

types = Counter(p['schema_type'] for p in pages)
tlds = Counter()
for p in pages:
    host = urlparse(p['url']).netloc
    if host.endswith('.co.uk'): tlds['.co.uk'] += 1
    elif host.endswith('.com.au'): tlds['.com.au'] += 1
    elif host.endswith('.co.nz'): tlds['.co.nz'] += 1
    else: tlds[f'.{host.rsplit(".",1)[-1]}'] += 1

print(f'Total pages: {len(pages):,}')
print(f'\nBy type:')
for t, n in types.most_common():
    print(f'  {t:25s} {n:5,}')
print(f'\nBy TLD (top 5):')
for t, n in tlds.most_common(5):
    print(f'  {t:10s} {n:5,}')